In [1]:
# ============================================================
# FROM-SCRATCH SEMANTIC ENCODER, BI-ENCODER, AND CROSS ENCODER
# No external ML libraries used
# ============================================================


# ------------------------------------------------------------
# 1. Basic text cleaning
# ------------------------------------------------------------

def clean_text(text):
    """
    Convert text to lowercase and remove basic punctuation.
    This makes comparison more consistent.
    """
    text = text.lower()

    punctuation = [",", ".", "-", "_", "/", "(", ")", "[", "]", ":", ";"]
    for p in punctuation:
        text = text.replace(p, " ")

    words = text.split()
    return words


# ------------------------------------------------------------
# 2. Semantic dictionary
# ------------------------------------------------------------
# This is our tiny "meaning brain".
# In real AI systems, these meanings are learned from huge data.
# Here we manually define them to understand the concept.

SEMANTIC_GROUPS = {
    "apple_brand": [
        "apple", "iphone", "ios", "macbook"
    ],
    "samsung_brand": [
        "samsung", "galaxy"
    ],
    "phone": [
        "phone", "mobile", "smartphone", "iphone", "galaxy"
    ],
    "laptop": [
        "laptop", "notebook", "macbook"
    ],
    "premium": [
        "pro", "max", "ultra", "premium"
    ],
    "storage": [
        "gb", "128gb", "256gb", "512gb", "1tb", "storage"
    ],
    "data_governance": [
        "data", "governance", "stewardship", "policy", "data-quality"
    ],
    "duplicate_matching": [
        "duplicate", "matching", "match", "entity", "resolution", "dedupe"
    ],
    "quality": [
        "quality", "accuracy", "completeness", "validity"
    ]
}


# ------------------------------------------------------------
# 3. Semantic Encoder
# ------------------------------------------------------------

def semantic_encoder(text):
    """
    Convert text into a meaning vector.

    Example:
    "Apple iPhone 15 Pro Max 256GB"
    may become:
    {
        "apple_brand": 1,
        "phone": 1,
        "premium": 1,
        "storage": 1
    }

    This is a simplified semantic encoder.
    """
    words = clean_text(text)

    vector = {}

    for concept, keywords in SEMANTIC_GROUPS.items():
        score = 0

        for word in words:
            if word in keywords:
                score += 1

        # Store concept score
        vector[concept] = score

    return vector


# ------------------------------------------------------------
# 4. Cosine similarity from scratch
# ------------------------------------------------------------

def cosine_similarity(vec1, vec2):
    """
    Compare two vectors using cosine similarity.

    Cosine similarity tells us if two vectors point in similar directions.

    Score range:
    0   = not similar
    1   = very similar
    """
    dot_product = 0
    length_vec1 = 0
    length_vec2 = 0

    for key in vec1:
        dot_product += vec1[key] * vec2.get(key, 0)
        length_vec1 += vec1[key] * vec1[key]

    for key in vec2:
        length_vec2 += vec2[key] * vec2[key]

    if length_vec1 == 0 or length_vec2 == 0:
        return 0

    return dot_product / ((length_vec1 ** 0.5) * (length_vec2 ** 0.5))


# ------------------------------------------------------------
# 5. Bi-Encoder
# ------------------------------------------------------------

def bi_encoder_score(text1, text2):
    """
    Bi-encoder approach:
    1. Encode text1 separately
    2. Encode text2 separately
    3. Compare vectors
    """
    vector1 = semantic_encoder(text1)
    vector2 = semantic_encoder(text2)

    score = cosine_similarity(vector1, vector2)

    return score, vector1, vector2


# ------------------------------------------------------------
# 6. Helper for exact token overlap
# ------------------------------------------------------------

def token_overlap_score(text1, text2):
    """
    Simple word overlap score.
    """
    words1 = set(clean_text(text1))
    words2 = set(clean_text(text2))

    if len(words1) == 0 or len(words2) == 0:
        return 0

    common = words1.intersection(words2)
    total = words1.union(words2)

    return len(common) / len(total)


# ------------------------------------------------------------
# 7. Cross Encoder
# ------------------------------------------------------------

def cross_encoder_score(text1, text2):
    """
    Cross-encoder style comparison.

    Key idea:
    Instead of encoding text1 and text2 separately,
    it looks at both texts together and uses interaction features.

    This is NOT a real neural cross encoder.
    It is a simple from-scratch teaching version.
    """

    words1 = clean_text(text1)
    words2 = clean_text(text2)

    set1 = set(words1)
    set2 = set(words2)

    # Feature 1: semantic similarity using our semantic encoder
    semantic_score, vector1, vector2 = bi_encoder_score(text1, text2)

    # Feature 2: exact token overlap
    overlap_score = token_overlap_score(text1, text2)

    # Feature 3: brand agreement
    # If both are Apple, good.
    # If one is Apple and the other Samsung, penalty.
    apple_1 = vector1["apple_brand"] > 0
    apple_2 = vector2["apple_brand"] > 0
    samsung_1 = vector1["samsung_brand"] > 0
    samsung_2 = vector2["samsung_brand"] > 0

    brand_score = 0

    if apple_1 and apple_2:
        brand_score = 1
    elif samsung_1 and samsung_2:
        brand_score = 1
    elif (apple_1 and samsung_2) or (samsung_1 and apple_2):
        brand_score = -1
    else:
        brand_score = 0

    # Feature 4: product type agreement
    phone_1 = vector1["phone"] > 0
    phone_2 = vector2["phone"] > 0
    laptop_1 = vector1["laptop"] > 0
    laptop_2 = vector2["laptop"] > 0

    type_score = 0

    if phone_1 and phone_2:
        type_score = 1
    elif laptop_1 and laptop_2:
        type_score = 1
    elif (phone_1 and laptop_2) or (laptop_1 and phone_2):
        type_score = -1
    else:
        type_score = 0

    # Feature 5: important number match
    # In product matching, numbers like 15, 256gb, 512gb matter.
    numbers1 = []
    numbers2 = []

    for word in words1:
        if contains_digit(word):
            numbers1.append(word)

    for word in words2:
        if contains_digit(word):
            numbers2.append(word)

    number_score = 0
    if len(numbers1) > 0 and len(numbers2) > 0:
        common_numbers = set(numbers1).intersection(set(numbers2))
        if len(common_numbers) > 0:
            number_score = 1
        else:
            number_score = -1

    # Weighted formula
    # Cross encoder can use interactions and business rules.
    raw_score = (
        0.40 * semantic_score +
        0.20 * overlap_score +
        0.20 * brand_score +
        0.15 * type_score +
        0.05 * number_score
    )

    # Keep score between 0 and 1
    if raw_score < 0:
        raw_score = 0
    if raw_score > 1:
        raw_score = 1

    return raw_score


def contains_digit(word):
    """
    Check if a word contains any number.
    """
    for ch in word:
        if ch >= "0" and ch <= "9":
            return True
    return False


# ------------------------------------------------------------
# 8. Demo examples
# ------------------------------------------------------------

records = [
    "Apple iPhone 15 Pro Max 256GB",
    "iPhone 15 Pro Max by Apple with 256GB storage",
    "Samsung Galaxy S24 Ultra 256GB",
    "Apple MacBook Pro laptop 512GB",
    "Data Governance and Data Quality Framework",
    "Enterprise data stewardship and quality policy"
]

query = "Apple smartphone iPhone 15 Pro 256GB"


print("QUERY:")
print(query)
print()


# ------------------------------------------------------------
# 9. Bi-encoder search
# ------------------------------------------------------------

print("BI-ENCODER RESULTS")
print("==================")

bi_results = []

for record in records:
    score, q_vec, r_vec = bi_encoder_score(query, record)
    bi_results.append((record, score))

# Sort highest score first
bi_results.sort(key=lambda x: x[1], reverse=True)

for record, score in bi_results:
    print(round(score, 3), "->", record)


print()
print("CROSS-ENCODER RESULTS")
print("=====================")

cross_results = []

for record in records:
    score = cross_encoder_score(query, record)
    cross_results.append((record, score))

cross_results.sort(key=lambda x: x[1], reverse=True)

for record, score in cross_results:
    print(round(score, 3), "->", record)


# ------------------------------------------------------------
# 10. Show semantic vector example
# ------------------------------------------------------------

print()
print("SEMANTIC VECTOR EXAMPLE")
print("=======================")

sample_text = "Apple iPhone 15 Pro Max 256GB"
sample_vector = semantic_encoder(sample_text)

print(sample_text)
print(sample_vector)

QUERY:
Apple smartphone iPhone 15 Pro 256GB

BI-ENCODER RESULTS
0.9 -> Apple iPhone 15 Pro Max 256GB
0.877 -> iPhone 15 Pro Max by Apple with 256GB storage
0.6 -> Apple MacBook Pro laptop 512GB
0.478 -> Samsung Galaxy S24 Ultra 256GB
0.0 -> Data Governance and Data Quality Framework
0.0 -> Enterprise data stewardship and quality policy

CROSS-ENCODER RESULTS
0.903 -> Apple iPhone 15 Pro Max 256GB
0.851 -> iPhone 15 Pro Max by Apple with 256GB storage
0.284 -> Apple MacBook Pro laptop 512GB
0.211 -> Samsung Galaxy S24 Ultra 256GB
0.0 -> Data Governance and Data Quality Framework
0.0 -> Enterprise data stewardship and quality policy

SEMANTIC VECTOR EXAMPLE
Apple iPhone 15 Pro Max 256GB
{'apple_brand': 2, 'samsung_brand': 0, 'phone': 1, 'laptop': 0, 'premium': 2, 'storage': 1, 'data_governance': 0, 'duplicate_matching': 0, 'quality': 0}
